In [ ]:
# קריאת קובץ PCAP

In [ ]:
import pyshark
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model


In [ ]:
# 1️⃣ פונקציה לניתוח קובץ PCAP ולחילוץ נתונים
def extract_traffic_features(pcap_file):
    capture = pyshark.FileCapture(pcap_file)
    packet_sizes = []
    arrival_times = []
    
    start_time = None
    for packet in capture:
        if hasattr(packet, 'length') and hasattr(packet, 'sniff_time'):
            packet_sizes.append(int(packet.length))
            if start_time is None:
                start_time = packet.sniff_time.timestamp()
            arrival_times.append(packet.sniff_time.timestamp() - start_time)
    
    capture.close()
    return packet_sizes, arrival_times


In [ ]:
#יצירת תמונת FlowPic

In [ ]:
# 2️⃣ פונקציה להמרת הנתונים לתמונה (FlowPic)
def create_flowpic(packet_sizes, arrival_times, img_size=(32, 32)):
    matrix = np.zeros(img_size)
    norm_sizes = np.interp(packet_sizes, (min(packet_sizes), max(packet_sizes)), (0, img_size[0]-1))
    norm_times = np.interp(arrival_times, (min(arrival_times), max(arrival_times)), (0, img_size[1]-1))
    
    for i in range(len(packet_sizes)):
        x, y = int(norm_sizes[i]), int(norm_times[i])
        matrix[x, y] += 1
    
    plt.imshow(matrix, cmap='hot', interpolation='nearest')
    plt.axis('off')
    plt.savefig("flowpic.png")
    plt.close()
    return np.expand_dims(matrix, axis=(0, -1))  # הוספת מימד לתמונת הקלט


In [ ]:
#רשת CNN בסיסית

In [ ]:
# 3️⃣ פונקציה לסיווג עם רשת CNN
def classify_flowpic(model_path, flowpic):
    model = load_model(model_path)
    prediction = model.predict(flowpic)
    categories = ['Browsing', 'Chat', 'VoIP', 'Video', 'File Transfer']  # קטגוריות לדוגמה
    predicted_category = categories[np.argmax(prediction)]
    return predicted_category

In [ ]:
# 4️⃣ פונקציה ראשית להפעלת כל השלבים יחד
def analyze_pcap(pcap_file, model_path):
    print("🔍 ניתוח קובץ PCAP...")
    packet_sizes, arrival_times = extract_traffic_features(pcap_file)
    print("📸 יצירת FlowPic...")
    flowpic = create_flowpic(packet_sizes, arrival_times)
    print("🤖 סיווג התעבורה...")
    category = classify_flowpic(model_path, flowpic)
    print(f"✅ הקובץ מסווג כ: {category}")
    return category

# דוגמה להפעלה:
analyze_pcap("example.pcap", "flowpic_model.h5")
